In [ ]:
# الخطوة 1: تحميل البيانات من Kaggle
import kagglehub, pandas as pd, os, re, warnings
warnings.filterwarnings('ignore')

path = kagglehub.dataset_download("nalgondalokesh/drug-food-interaction-dataset")
df = pd.read_csv(os.path.join(path, "drug_food_interactions.csv"))
print(f"✅ البيانات جاهزة: {df.shape[0]} صف")
print(df.head(3))

In [ ]:
# الخطوة 2: NLP - تصنيف نوع التداخل من النص
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# تنظيف النص
df['clean'] = df['food_interactions'].fillna('').astype(str).str.lower()
df['clean'] = df['clean'].str.replace(r'[^a-z ]+', ' ', regex=True).str.strip()

# تصنيف تلقائي بالكلمات المفتاحية
def label(t):
    t = str(t).lower()
    if not t.strip(): return 'No Interaction'
    if any(w in t for w in ['avoid','do not',"don't",'contraindicated','dangerous','toxic','fatal']): return 'Avoid'
    if any(w in t for w in ['increase','decrease','reduce','absorption','level','affect','alter']): return 'Monitor'
    if any(w in t for w in ['with food','with meal','with milk','with water','take with']): return 'Take With Food'
    if any(w in t for w in ['limit','moderate','caution','grapefruit','caffeine','alcohol']): return 'Limit'
    return 'General Interaction'

df['label'] = df['food_interactions'].apply(label)
print("📊 توزيع الفئات:")
print(df['label'].value_counts())

df['label'].value_counts().plot(kind='bar', color=['#e74c3c','#f39c12','#2ecc71','#3498db','#9b59b6'])
plt.title('توزيع فئات التداخل الدوائي الغذائي')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# الخطوة 3: تدريب نموذج NLP (TF-IDF + Logistic Regression)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df['label'])
X = df['clean']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"🏋️ تدريب: {len(X_train)} | 🧪 اختبار: {len(X_test)}")

# النموذج: TF-IDF + Logistic Regression
model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n✅ دقة النموذج: {acc:.2%}")
print("\n📋 تقرير تفصيلي:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# الخطوة 4: تجربة تنبؤ جديد
def predict(drug_name, food_text):
    clean = re.sub(r'[^a-z ]+', ' ', food_text.lower()).strip()
    pred_enc = model.predict([clean])[0]
    pred_label = le.inverse_transform([pred_enc])[0]
    proba = model.predict_proba([clean])[0].max()
    icons = {'Avoid': '🔴', 'Monitor': '🟡', 'Take With Food': '🟢', 'Limit': '🟠', 'General Interaction': '🔵', 'No Interaction': '⚪'}
    print("="*50)
    print(f"💊 الدواء: {drug_name}")
    print(f"🔮 نوع التداخل: {icons.get(pred_label,'❓')} {pred_label}")
    print(f"📊 الثقة: {proba:.1%}")
    print("="*50)
    return pred_label

# مثال:
predict("Warfarin", "avoid grapefruit and leafy vegetables")
predict("Metformin", "take with food to reduce stomach upset")
predict("Ibuprofen", "limit alcohol consumption")